# Module 2: Cyber Threat Landscape, Data Sources, Pre-processing, and First ML Model

---

## Learning Objectives

By the end of this module, you will be able to:

- Describe the major attack types, and place the steps of an attack on the **Cyber Kill
  Chain** and in **MITRE ATT&CK**.
- Compare the major public security datasets and say which one fits a given problem.
- Describe the **machine learning workflow** end to end: framing, cleaning, feature
  engineering, modelling, evaluation.
- Clean a genuinely messy log — duplicates, missing values, corrupt entries, inconsistent
  capitalisation, and values that are physically impossible.
- Engineer features from raw log fields that a model can actually use.
- Say what a **logistic regression** computes, train one, and read a classification report
  and a confusion matrix.
- **Recognise a result that is too good to be true, and diagnose why.** You will train a
  model that scores a perfect 100% and then show it learned nothing worth having.
- **Act on an audit rather than just performing one** — take the defects Module 1 found in
  a real dataset, fix them, and then *measure* whether fixing them changed anything.

---

## Where we left off

Module 1 audited a dataset and trained nothing. This module trains your first two models.

The first half is reading: what attackers do, how defenders describe it, what data exists,
and the steps between a raw log and a trained model. It is shorter than it looks and it
contains two short activities. The second half is two labs. In Lab 1 you clean a small,
deliberately broken firewall log and train a classifier that scores 100%. In Lab 2 you go
back to Module 1's 88,647 URLs, act on what the audit found, and get a lower score that is
worth far more.

---

## 1. What attackers do

A detector is built to find something specific, so it helps to have names for the things.
These six categories cover most of what you will read about, and the last column says where
each leaves evidence, because that decides what data a detector would need.

| category | what it is | where it leaves evidence |
|---|---|---|
| **Malware** | software written to damage, spy or extort: ransomware, trojans, spyware | files on disk, process behaviour, outbound connections |
| **Phishing and social engineering** | deceiving a person into giving up credentials or running something | email content, URLs, login pages |
| **Denial of service (DoS / DDoS)** | flooding a service until legitimate users cannot reach it, often from a botnet | network traffic volume and source addresses |
| **Man-in-the-middle** | intercepting or altering traffic between two parties | network paths, certificates, DNS answers |
| **Web application attacks** | exploiting flaws in a site's code, such as SQL injection or cross-site scripting | web server and application logs |
| **Insider threats** | misuse of legitimate access, deliberate or accidental | access logs, usually with nothing technically "wrong" in them |

Two of these come up repeatedly in this course. Phishing is the subject of the URL data in
Modules 1 to 3 and the email data in Modules 5 and 6. Network flows from compromised
devices are the subject of Module 5's final lab and Module 7.

---

## 2. Describing an attack: the Kill Chain and ATT&CK

An attack is a sequence of steps, and defenders have two widely used ways of writing the
sequence down.

### The Cyber Kill Chain

Lockheed Martin's model describes an intrusion as seven stages in order.

| stage | what the attacker is doing |
|---|---|
| 1. Reconnaissance | learning about the target |
| 2. Weaponisation | building the payload, for instance malware inside a document |
| 3. Delivery | getting it to the target, for instance by email |
| 4. Exploitation | triggering it |
| 5. Installation | establishing a foothold that survives a reboot |
| 6. Command and control (C2) | opening a channel back to the attacker |
| 7. Actions on objectives | doing what they came to do |

The model's point is that the attacker has to succeed at every stage and the defender needs
to break only one.

### MITRE ATT&CK

The Kill Chain is a single path. **ATT&CK** is a catalogue. It is a public knowledge base,
built from observed intrusions, organised on three levels:

- **Tactics** — the attacker's goal at a given moment: *Initial Access*, *Persistence*,
  *Exfiltration*. The Enterprise matrix has fourteen.
- **Techniques** — a way of achieving a tactic. *Phishing* (T1566) is a technique for
  Initial Access. Each has an ID.
- **Procedures** — a specific real-world use of a technique: *this group sent a
  spearphishing email with a malicious PDF to the finance department.*

Analysts use it as a shared vocabulary, so that "T1053" means the same thing in every
report. The matrix is at <https://attack.mitre.org/>.

### Why a machine learning course cares

A model does not detect "an attack". It detects one technique, at one stage, in one kind of
data. The frameworks are how you say which. The URL classifier you will train in Lab 2 looks
at *Delivery* by *Phishing* and at nothing else: it would not notice the malware that a
successful phish goes on to install, because that leaves no trace in a URL. When you read a
claim about a detector, the first useful question is which technique it covers, and the
frameworks give you the words to ask it. Module 7 uses the sister framework, MITRE ATLAS,
for attacks on the models themselves.

### Activity: map an attack to ATT&CK

**Scenario.** An employee receives a well-crafted email that appears to be from their IT
department, asking them to update their password. The employee clicks the link, enters
their credentials on a fake login page, and is then redirected to a page that downloads a
file named `Update.exe`. The employee runs the file, which installs malware that sets up a
scheduled task to run every hour, ensuring it survives a reboot.

Using the Enterprise matrix at <https://attack.mitre.org/>, complete steps 2 to 4. Step 1
is done for you.

| step | what happened | tactic | technique | technique ID |
|---|---|---|---|---|
| 1 | phishing email is sent and opened | Initial Access | Phishing | T1566 |
| 2 | the user's credentials are stolen on the fake page | **[insert answer]** | **[insert answer]** | **[insert answer]** |
| 3 | the user is tricked into running `Update.exe` | **[insert answer]** | **[insert answer]** | **[insert answer]** |
| 4 | the malware creates a scheduled task | **[insert answer]** | **[insert answer]** | **[insert answer]** |

**Question.** The attacker had to succeed at all four steps. If you could add monitoring for
only **one** of the techniques you identified, which would you choose, and what data would
that monitoring need to collect? Justify your choice.

**[insert answer]**

*(Stretch.)* **Two further ATT&CK exercises, if you want more practice.**

**A second scenario.** A company's e-commerce site becomes unavailable. Its web servers are
being flooded with requests from thousands of addresses around the world. Find the tactic,
technique and ID. Then say what is fundamentally different about this attacker's goal
compared with the phishing scenario.

**[insert answer]**

**Comparing two threat groups.** MITRE's ATT&CK Navigator
(<https://mitre-attack.github.io/attack-navigator/>) lets you overlay the techniques used by
different groups. Follow MITRE's exercise sheet,
<https://attack.mitre.org/docs/training-cti/Comparing%20Layers%20in%20Navigator.pdf>, for
APT39 and OceanLotus, and report: one technique unique to each group, the tactic with the
most overlap, and the five techniques you would defend first on a limited budget.

**[insert answer]**

---

## 3. Data: what exists, and what this course uses

A model learns from examples, so the first practical question about any detection problem
is whether suitable examples exist. Four public datasets come up constantly in security-ML
papers and job interviews. It is worth knowing what each one is *for*, because a model
trained on the wrong data cannot be repaired by better modelling.

| dataset | what it contains | built for |
|---|---|---|
| **CICIDS2017 / CSE-CIC-IDS2018** | labelled network flows from a test network with simulated attacks: brute force, DoS, web attacks, infiltration | network intrusion detection |
| **PhishTank** | a community-verified, continuously updated list of phishing URLs | phishing URL detection, as a source of positive examples |
| **EMBER** | features extracted from over a million Windows executables, labelled malicious or benign | static malware detection: judging a file without running it |
| **IoT-23** | network captures from infected and clean IoT devices | detecting compromised IoT devices |

**This course does not use those four.** They are the landscape you should recognise. What
you will actually work with is hosted with the course and downloads automatically:

| data | modules |
|---|---|
| 88,647 phishing and legitimate URLs, as 111 lexical features | 1, 2, 3 |
| a 30-row firewall log, built to be broken | 2 |
| 284,807 credit card transactions, 492 fraudulent | 3, 4, 5 |
| DGA and legitimate domain names; malware binaries as images; 39,154 labelled emails | 5 (and 6, 7) |
| IoT network flows: port scans and backdoor traffic | 5, 7 |
| a week of synthetic logs from eight hosts, with ground truth | 6 |

### Activity: match the data to the problem

For each scenario, say which of the four public datasets above would be the most relevant
for training a detector, and **why**. The reason is the answer; the name alone is not.

**Scenario A — the sluggish network.** An internal network has become extremely slow and the
main web server has crashed twice in an hour. You suspect a device inside is flooding the
network, but you do not know which, or with what.

**[insert answer]**

**Scenario B — the suspicious executable.** An employee extracted `invoice_printer.exe` from
an emailed ZIP file and did not run it. You have the file and need to judge whether it is
malware.

**[insert answer]**

**Scenario C — the rogue camera.** A smart security camera on the corporate network is
sending an unusually large amount of data to an unknown address overseas.

**[insert answer]**

*(Stretch.)* For one of your three answers, name something about the scenario that the
dataset you chose would **not** prepare a model for.

**[insert answer]**

---

## 4. From a question to a model: the workflow

Every project in this course, and nearly every one outside it, goes through the same six
stages. The labs below walk the first five.

1. **Frame the question.** "Can we use AI to stop hackers?" cannot be answered. "Given the
   features of a URL, can we predict whether it is a phishing link?" can: it names the
   input, the output and what success would look like.
2. **Prepare the data.** Real data is incomplete, inconsistent and occasionally impossible.
   This stage is most of the work, and Section 5 breaks it down.
3. **Choose a model.** Labelled examples and a category to predict call for a classifier.
   This module uses the simplest one worth using, logistic regression, and introduces it
   properly just before you train it.
4. **Train.** The model adjusts its internal numbers to fit a portion of the data set aside
   for the purpose, the **training set**.
5. **Evaluate.** Measure it on rows it has never seen, the **test set**. Accuracy alone is
   rarely enough; Lab 1.6 introduces precision and recall.
6. **Deploy and monitor.** A model in production meets data that drifts away from what it
   was trained on, and adversaries who adapt to it. This course returns to that in Module 7.

## 5. Preparing data

Four jobs, usually in this order.

**Cleaning.** Remove exact duplicates. Decide what to do about missing values: drop the
row, or fill the gap with something defensible such as the column's median. Find the values
that cannot be true — a port number of 99999, a timestamp that is not a date — and treat
them as missing. Make text consistent, so that `Allow` and `ALLOW` are one value, not two.

**Encoding.** A model does arithmetic, so text has to become numbers. For a column of
categories with no order — protocol is `TCP`, `UDP` or `ICMP` — the standard method is
**one-hot encoding**: one new 0/1 column per category. Mapping them to 1, 2, 3 instead would
tell the model that ICMP is three times TCP, which means nothing.

**Scaling.** Features arrive in very different units. A packet count runs into the
thousands and a 0/1 flag does not, and many models are swayed by the larger numbers simply
because they are larger. Scaling puts features on a comparable footing. You will use two
scalers in this module:

- **`MinMaxScaler`** squeezes each column into the range 0 to 1. It is easy to read, and it
  suits Lab 1, where the columns are well behaved.
- **`StandardScaler`** shifts each column to an average of 0 and a spread of 1. It suits
  Lab 2, where many columns are counts with a few very large values: under min-max scaling,
  one enormous value would press every other row into a sliver near zero.

Either way, the scaler must learn its numbers **from the training set only**. Lab 1.5
explains why, and Module 3 measures what it costs to get that wrong.

**Feature engineering.** Sometimes the useful quantity is not a column but something you
can compute from one. An IP address as text is useless to a model. *Whether that address is
inside the company network* is a 0/1 fact it can use. A timestamp is useless; *the hour of
day* may not be. In Lab 1 you engineer four features like that from raw log fields. In
Lab 2 you will not need to: the URL dataset's 111 columns are features its authors already
engineered from each URL, which is what Module 1's column families were.

---

# Lab 1: Clean a Broken Log

Everything above was reading. From here you run code.

The dataset is a **firewall log with 30 rows**, constructed for this course. Thirty. You will be able to read every row
on screen, which is deliberate and is the whole reason we start here — when you can see the
entire dataset, you can *verify* that your cleaning did what you intended instead of hoping.

It is also deliberately broken. Every defect in it is one that appears constantly in real
logs, and you will fix each one:

| defect | what it looks like |
|---|---|
| duplicate rows | the same event logged twice |
| missing values | empty fields across several columns |
| a corrupt entry | a timestamp reading `2023-&*-32 0^$@%^&#` |
| an impossible value | a destination port of `99999`, when the maximum is 65535 |
| inconsistent capitalisation | `Allow`, `ALLOW`, `Deny` and `deny` in one column |
| wrong types | timestamps stored as text, counts stored as decimals |

Later in the module you will take everything you learn here and apply it to a dataset of
**88,647 rows** — where you cannot eyeball anything and have to trust your commands.

In [1]:
# ============================================================
# Module 2 Setup -- run this cell first, every session.
# ============================================================
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

DATA_URL = "https://github.com/abramweigant/AI-Cyber-Intro-Cert/raw/refs/heads/main/"

print(f"pandas {pd.__version__} | numpy {np.__version__} | seed {SEED}")

pandas 3.0.6 | numpy 2.5.3 | seed 42


## Lab 1.1 — Load and inspect

Four commands tell you the state of a new table. Run them and *read* the output.

In [2]:
log = pd.read_csv(DATA_URL + "Log_Data.csv")
print(f"loaded {log.shape[0]} rows, {log.shape[1]} columns\n")
display(log.head(10))

loaded 30 rows, 11 columns



,timestamp,source_ip,destination_ip,source_port,destination_port,protocol,packet_count,byte_count,firewall_rule,device_type,threat_type
0,2023-09-01 08:00:15,192.168.1.10,104.16.124.96,54321,443.0,TCP,10.0,8540.0,Allow,Laptop,Benign
1,2023-09-01 08:01:22,192.168.1.15,172.217.5.142,61234,80.0,TCP,5.0,3450.0,Allow,Server,Benign
2,2023-09-01 08:02:05,103.27.14.88,192.168.1.25,12345,22.0,TCP,40.0,2800.0,Deny,Unknown,Brute Force
3,2023-09-01 08:02:06,103.27.14.88,192.168.1.25,12346,22.0,TCP,42.0,2940.0,Deny,Unknown,Brute Force
4,2023-09-01 08:02:07,103.27.14.88,192.168.1.25,12347,22.0,TCP,38.0,2660.0,Deny,Unknown,Brute Force
5,2023-09-01 08:03:10,192.168.1.10,8.8.8.8,51000,53.0,UDP,2.0,140.0,ALLOW,Laptop,Benign
6,2023-09-01 08:04:30,203.0.113.55,192.168.1.50,8080,8080.0,TCP,1.0,40.0,Deny,IoT Device,Scan
7,2023-09-01 08:05:01,192.168.1.15,192.168.1.10,80,54321.0,TCP,12.0,9870.0,Allow,Server,Benign
8,2023-09-01 08:05:01,192.168.1.15,192.168.1.10,80,54321.0,TCP,12.0,9870.0,Allow,Server,Benign
9,2023-09-01 08:05:45,192.168.1.20,NaN,137,137.0,UDP,4.0,280.0,Allow,NaN,Benign


In [ ]:
# --- TASK 1.0 -- INSPECT BEFORE YOU TOUCH ANYTHING ---
# Run all four and read the output:
#   1. .info()            -- dtypes and non-null counts
#   2. .isnull().sum()    -- missing values per column
#   3. .duplicated().sum()-- exact duplicate rows
#   4. .describe()        -- ranges. Look hard at max destination_port.
#
# Four separate defects are visible in this output. Find as many as you can.

# YOUR CODE HERE

### What that told you

Read it back carefully, because every one of these matters:

- **`timestamp` is not a datetime** — pandas is treating it as text (shown as `object` on
  pandas 2 and `str` on pandas 3). You cannot ask a
  string what hour it happened at.
- **Eight columns have missing values**, including `threat_type` — which is the thing we are
  trying to predict. A row with no answer cannot teach a model anything.
- **There are 2 duplicate rows.** The same event, logged twice, would count twice.
- **`destination_port` has a maximum of 99999.** Ports are 16-bit; the largest possible
  value is 65535. That row is not unusual, it is *wrong*.

**Task 1.1.** One defect is not visible in any of the output above: a corrupt value hiding
in a text column. Find it. Print the distinct values of every text column and read them.

*Hint: `log.select_dtypes(include=["object", "string"]).columns` gives you the text columns, and
`.value_counts()` on each shows what is in them.*

In [ ]:
# --- TASK 1.1 ---
# Print the distinct values of every text (object) column and read them.
# Two defects are hiding in this output. One is obvious; one is not.
#
# Hint: log.select_dtypes(include=["object", "string"]).columns gives you the text columns.
# (Asking for both names works on every recent pandas version.)

# YOUR CODE HERE

## Lab 1.2 — Clean it

You now know every defect. Fix them.

**Task 1.2.** Produce a cleaned DataFrame called `clean`. Work in this order, because order
matters:

1. Drop duplicate rows.
2. Remove the row with the corrupt timestamp.
3. Drop rows where `destination_ip` **or** `threat_type` is missing. Both are unrecoverable:
   one is a critical feature, the other is the answer.
4. Convert `timestamp` to a real datetime with `pd.to_datetime`.
5. Remove rows where `destination_port` exceeds 65535.
6. Fill the remaining missing text values with `'Unknown'`.
7. Normalise the capitalisation of `firewall_rule`.
8. Convert `destination_port`, `packet_count` and `byte_count` to integers.

Then print how many rows survived, and confirm there are no nulls left.

> **On step 3 versus step 6 — this is the judgement call.** We *delete* rows missing the
> destination IP or the label, and we *keep* rows missing the device type, filling them with
> `'Unknown'`. The difference is whether the missing thing can be replaced by an honest
> placeholder. "I do not know what kind of device this was" is a true statement you can
> encode. "I do not know what the answer is" is not something you can teach a model.

In [ ]:
# --- TASK 1.2 ---
# Build a cleaned DataFrame called `clean`, following the eight steps above IN ORDER.
# Order matters: converting to datetime before removing the corrupt row will raise,
# and astype(int) before fillna will raise on NaN.
#
# Finish by printing the surviving row count and confirming no nulls remain.

clean = log.copy()

# YOUR CODE HERE

In [6]:
# ============================================================
# CHECKPOINT -- run after Task 1.2. Verifies the cleaned log's state;
# these are dataset facts and exact on every machine. A failure means
# one of the eight steps went wrong, or ran in the wrong order.
# ============================================================
try:
    assert len(clean) == 24, f"{len(clean)} rows survived, expected 24"
    assert clean.isnull().sum().sum() == 0, "nulls remain after cleaning"
    assert str(clean['timestamp'].dtype).startswith('datetime64'), \
        f"timestamp is {clean['timestamp'].dtype}, expected datetime64"
    assert clean['firewall_rule'].value_counts().to_dict() == {'Allow': 12, 'Deny': 12}, \
        "firewall_rule has not collapsed to exactly Allow 12 / Deny 12"
    assert clean['destination_port'].max() <= 65535, "an impossible port survived"
except NameError:
    print("Checkpoint skipped: complete Task 1.2 first -- it must produce a DataFrame called `clean`.")
else:
    print("Checkpoint passed: 24 clean rows, real datetimes, Allow 12 / Deny 12.")


Checkpoint passed: 24 clean rows, real datetimes, Allow 12 / Deny 12.


## Lab 1.3 — Feature engineering

A model cannot use a timestamp or an IP address. It can use *what you extract from them*,
and choosing what to extract is where security knowledge becomes a technical advantage.

**Task 1.3.** Add four features to `clean`:

| feature | definition | why a security analyst would want it |
|---|---|---|
| `hour_of_day` | hour from the timestamp | attackers work outside business hours |
| `day_of_week` | Monday=0 … Sunday=6 | weekend activity has a different baseline |
| `is_internal_traffic` | 1 if source **and** destination are `192.168.` | internal-to-internal has a different risk profile |
| `bytes_per_packet` | `byte_count / packet_count` | scans send many tiny packets; transfers send few large ones |

Guard the division: a row with zero packets would give you infinity.

In [ ]:
# --- TASK 1.3 ---
# Add four features to `clean`: hour_of_day, day_of_week, is_internal_traffic,
# and bytes_per_packet. Guard the division so a zero-packet row cannot produce inf.

# YOUR CODE HERE

## Lab 1.4 — Correlation

Before modelling, look at how the numeric features relate to each other and to the label.

In [ ]:
# --- TASK 1.4 -- CORRELATION HEATMAP ---
# 1. Select only the numeric columns of your cleaned DataFrame.
# 2. Compute the Pearson correlation matrix.
# 3. Draw it as an annotated heatmap, centred on zero.
#
# Then ask yourself how much you should trust a correlation computed on 24 rows.

import seaborn as sns
import matplotlib.pyplot as plt

# YOUR CODE HERE

**Task 1.4 — write-up.**

1. Which pair of features is most strongly correlated, and is that surprising given how you
   built them?
2. `hour_of_day` takes only four distinct values in this dataset. What does that do to any
   correlation you compute with it, and would you trust the number?
3. Normally we drop one of a pair of highly correlated features. Here we are keeping
   everything. Given the size of this dataset, argue whether that is defensible.

**[insert response]**

## Lab 1.5 — Prepare for modelling

Three steps, in a fixed order, and the order is the point.

1. **Drop what you cannot use.** `timestamp`, `source_ip` and `destination_ip` are text or
   dates; you already extracted their value into features. Drop them.
2. **Encode the text.** One-hot encode `protocol`, `firewall_rule` and `device_type`. Label
   encode `threat_type` into integers.
3. **Split, then scale.** Fit the scaler on the training data *only*, then apply it to both.

> **Why split before scaling.** A scaler computes the minimum and maximum of each column. If
> you fit it on everything, the test rows influence those numbers, and information from your
> test set has leaked into your training pipeline. Your score comes out better than it should
> and you have no way to tell. This is the single most common leak in applied ML, and Module 3
> is largely about the family of mistakes it belongs to.

**Task 1.5.** Do all three. Print the class mapping, the resulting feature count, and the
train/test sizes.

In [ ]:
# --- TASK 1.5 ---
# 1. Drop 'timestamp', 'source_ip', 'destination_ip'.
# 2. One-hot encode 'protocol', 'firewall_rule', 'device_type' (drop_first=True, dtype=int).
#    Label encode 'threat_type' -- .astype('category').cat.codes gives you integers,
#    and .cat.categories gives you the names in the matching order.
# 3. train_test_split with test_size=0.2, random_state=SEED, stratify=y.
# 4. Fit a MinMaxScaler on the TRAINING data only, then transform both.
#
# Name your results X_train_scaled, X_test_scaled, y_train, y_test, X, class_names
# so the cells below work.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# YOUR CODE HERE

## Lab 1.6 — Train, predict, evaluate

### Meet your first model

A **logistic regression** is about the simplest classifier there is, which is why it comes
first: you can see everything it does.

For each class it keeps one number per feature, called a **weight** or **coefficient**. To
judge a row, it multiplies each feature value by its weight and adds the results up. A
large total is evidence for that class; a negative one is evidence against. The totals are
then converted into probabilities that add up to 1 across the classes, and the class with
the highest probability is the prediction.

**Training** is the search for good weights. `model.fit(X_train_scaled, y_train)` shows the
model the training rows together with their correct answers and adjusts the weights until
its predictions match those answers as closely as it can manage.

Three things about the cell below.

- **Why there is a test set.** In Lab 1.5 you set aside 20% of the rows before training.
  The model never sees them during `fit`. Scoring it on rows it has seen would tell you how
  well it remembers; scoring it on rows it has not tells you how well it predicts. Only the
  second is worth knowing.
- **Why the split was stratified.** `stratify=y` keeps the mix of classes the same on both
  sides. With 24 rows and three classes, a purely random split could easily leave the test
  set without any example of one class.
- **`model.predict`** returns the predicted class for each row. `model.coef_` holds the
  learned weights, one row per class, and you will read them in Lab 1.7.

### Reading the two outputs

The **classification report** has one line per class and four columns.

| column | the question it answers |
|---|---|
| **precision** | of the rows the model *called* this class, what fraction really were? |
| **recall** | of the rows that really *are* this class, what fraction did the model find? |
| **f1-score** | one number that is high only when precision and recall both are |
| **support** | how many test rows actually belong to this class |

**Accuracy**, further down, is simply the fraction of all test rows predicted correctly.

The **confusion matrix** is a grid of counts: each row is what a test row really was, each
column is what the model said. Correct predictions fall on the diagonal, and every mistake
is somewhere off it, where you can see which class was mistaken for which.

**Before you run the cell,** look at the `support` you are about to get: the test set has
five rows. Write down which accuracy values are even possible with five rows, and what you
expect the model to score. You will report that prediction in the Task 1.7 write-up.

In [ ]:
# ============================================================
# Run-me: train the model and evaluate it on the five test rows.
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("predicted:", y_pred)
print("actual   :", y_test.values)
print()
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion matrix -- 5 test rows')
plt.tight_layout(); plt.show()

## Lab 1.7 — A perfect score is a symptom

Every metric came back **1.00**. Precision, recall, F1, accuracy, all three classes.

In machine learning, a perfect score on the first attempt is not a success. It is a symptom,
and the correct response is to go looking for the cause. There are three candidates and it is
worth ruling them out one at a time.

**Was the process leaky?** No. We split before scaling, so the test rows never influenced the
scaler. That specific mistake was avoided.

**Is the test set big enough to mean anything?** No. Five rows. With five, the only
scores this arrangement can produce are multiples of 20%. Getting 5 out of 5 right is roughly
as informative as flipping a coin five times and calling it a trend.

**Did the model find a shortcut?** This is the interesting one, and we can check it directly
rather than speculating.

In [11]:
# ============================================================
# Run-me: read the weights the model learned, one list per class.
# ============================================================
print("coef_ shape:", model.coef_.shape, " <- one ROW PER CLASS, not one vector")
print()

importance = pd.DataFrame(model.coef_.T, index=X.columns, columns=class_names)

for name in class_names:
    top = importance[name].sort_values(ascending=False).head(4)
    print(f"strongest evidence FOR '{name}':")
    for feat, val in top.items():
        print(f"    {feat:26s} {val:+.3f}")
    print()

coef_ shape: (3, 16)  <- one ROW PER CLASS, not one vector

strongest evidence FOR 'Benign':
    protocol_UDP               +0.465
    source_port                +0.455
    device_type_Laptop         +0.400
    bytes_per_packet           +0.378

strongest evidence FOR 'Brute Force':
    packet_count               +0.981
    device_type_Unknown        +0.812
    firewall_rule_Deny         +0.463
    day_of_week                +0.071

strongest evidence FOR 'Scan':
    device_type_IoT Device     +0.755
    firewall_rule_Deny         +0.566
    hour_of_day                +0.237
    day_of_week                +0.051



### What the coefficients actually say

Read the three lists. The strongest single driver of **Brute Force** is `packet_count`, and
that one is *real* — a brute-force attempt genuinely does send far more packets than anything
else in this log. Keep it in mind, because it is the honest signal here. Now look at what sits
directly underneath it:

- **Brute Force** is also predicted by `device_type_Unknown` and `firewall_rule_Deny`.
- **Scan** is predicted by `device_type_IoT Device` and `firewall_rule_Deny`.
- **Benign** is predicted by ordinary things — protocol, port, laptop.

Now go back and look at the cleaned data. In these 24 rows:

- **every** brute-force attempt came from a device whose type was missing
- **four of the five** scans came from an IoT device — and the fifth had no device type either
- **every** attack was denied by the firewall, and **every** benign event was allowed

The model did not learn what a brute-force attack looks like. It learned *"unknown device
type means brute force"*, and *"the firewall already blocked it, so it was bad."*

That second rule is worth sitting with. **The firewall's own verdict is in the feature set.**
We are asking a model to detect threats while handing it a column that says whether the
firewall already caught the threat. It is not detection, it is copying — and it will work
perfectly right up until it meets an attack the firewall allowed, which is the only kind of
attack anybody actually needs a model for.

### Memorisation versus generalisation

A model that **memorises** learns the specific artifacts of the examples it was shown. A model
that **generalises** learns a pattern that holds for examples it has never seen.

Ask what happens when this model meets a brute-force attack from a correctly-inventoried
laptop that the firewall allowed through. Every signal it relies on is absent. It will call
that traffic benign, confidently, and it will be wrong — while still reporting 100% accuracy
on the test set you gave it.

**Task 1.7 — write-up.**

1. A colleague looks at the cleaned log, notices that all seven brute-force rows share the
   source address `103.27.14.88`, and concludes: *"The model scored 100% because it
   memorised the attacker's IP."* Show from this notebook's own code that this cannot be
   what happened.
2. Of the three shortcuts above, which would be hardest to notice in a production dataset of
   a million rows, and why?
3. `firewall_rule` is the firewall's own decision about the traffic. Make the argument for
   removing it from the feature set — then make the argument for keeping it. Which do you
   find more convincing, and what would change your mind?
4. Before running Lab 1.6 you wrote down which accuracy values five test rows allow and what
   you expected the model to score. State your prediction as you wrote it, then compare it
   with what the model actually scored.
5. *(Stretch.)* You cannot make this dataset bigger. Name two things you *could* do to get a more honest
   estimate of how good this model really is.

**[insert response]**

---

# Lab 2: The Same Pipeline, On Real Data

Lab 1 taught you the mechanics on 24 rows you could read by eye. Now do it properly.

In **Module 1** you audited a dataset of 88,647 phishing URLs and found things wrong with it:

| what Module 1 found | scale |
|---|---|
| missing values encoded as `-1` rather than `NaN` | **66 of 111 columns** |
| duplicate rows | **1,438** |
| identical feature vectors carrying **both** labels | 6 vectors, 13 rows |
| columns with only one distinct value | **13** |

Auditing is only half the job. This lab is the other half: **act on the findings, build a
model, and then measure whether acting on them changed anything.**

That last step is the one people skip. It is easy to assume a defect must have mattered. The
honest move is to check.

In [12]:
urls = pd.read_parquet(DATA_URL + "phishing_dataset1.parquet")
print(f"loaded {urls.shape[0]:,} rows x {urls.shape[1]} columns")
print(f"label balance: {urls['phishing'].value_counts().to_dict()}")
print(f"phishing fraction: {urls['phishing'].mean():.3f}")

loaded 88,647 rows x 112 columns
label balance: {0: 58000, 1: 30647}
phishing fraction: 0.346


**Task 2.1.** Apply two of Module 1's findings, then train the same kind of model you
trained in Lab 1:

1. Drop the columns that carry no information (one distinct value).
2. Drop the exact duplicate rows.
3. Split 80/20, stratified, then scale with `StandardScaler` (Section 5 said why this
   scaler and not Lab 1's) — fitting it on the training set only.
4. Train a `LogisticRegression` and report accuracy, F1, and the classification report.
5. Report the **majority-class baseline** alongside it: the accuracy you would get by always
   guessing the more common class. A model that cannot beat that number has earned nothing.

**What you are deliberately not fixing.** Module 1's headline finding was the `-1`
sentinels in 66 columns, and this task leaves every one of them in place. A logistic
regression will take `-1` as just another number and use it. What it makes of them is the
first thing Module 3 investigates, and it is not what you would guess.

In [ ]:
# --- TASK 2.1 ---
# 1. Drop the columns with only one distinct value.
# 2. Drop exact duplicate rows.
# 3. Split 80/20 stratified, then scale (fit on train only).
# 4. Train a LogisticRegression(max_iter=2000) and report accuracy and F1.
# 5. Print the majority-class baseline next to your accuracy.
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

# YOUR CODE HERE

## Lab 2.2 — Did the cleanup actually help?

You dropped 13 columns and 1,438 rows because Module 1 showed they were defective. It is
natural to assume that helped.

**Task 2.2.** Find out. Train the same model three ways and compare:

1. On the cleaned data (you already have this number).
2. With the 13 constant columns **left in**.
3. With the duplicate rows **left in**.

Predict what you expect before you run it — and look back at the prediction you made in
Module 1, Task 5.2, question 4. Then write down what actually happened.

In [ ]:
# --- TASK 2.2 ---
# Train the same model three ways and print accuracy and F1 for each:
#   1. on your cleaned data
#   2. with the 13 constant columns left in
#   3. with the 1,438 duplicate rows left in
#
# Write your prediction down BEFORE you run it.
# Hint: a small helper function that takes a DataFrame and a label keeps this readable.

# YOUR CODE HERE

**Task 2.2 — write-up.**

1. What did you predict, and what happened?
2. Dropping the 13 dead columns changed accuracy by exactly nothing. Explain why a column
   with a single distinct value cannot affect a linear model's predictions.
3. *(Stretch.)* The duplicates made no meaningful difference here either. Describe a dataset or a model
   where the *same* 1,438 duplicates would have mattered enormously.
4. Given all that — was the audit a waste of time? Defend your answer.

**[insert response]**

---

## Module 2 Summary

You did two versions of the same job.

**On 24 rows** you cleaned a genuinely broken log — duplicates, a corrupt timestamp, an
impossible port, four spellings of two firewall verdicts, dates stored as text — engineered
four features from raw fields, trained a classifier, and scored a perfect 100%. Then you
opened up the model and found it had learned that unknown device types mean brute force and
that the firewall's own verdict predicts the answer. A perfect score, worth nothing.

**On 87,209 rows** you acted on an audit from Module 1, trained the same kind of model, and
got **93.35% against a 65.03% baseline**. A worse-looking number that is worth a great deal
more, because the 28-point gap over the baseline is real.

| | Lab 1 (toy log) | Lab 2 (real URLs) |
|---|---|---|
| rows after cleaning | 24 | 87,209 |
| test rows | 5 | 17,442 |
| accuracy | **100%** | 93.35% |
| majority baseline | 60% | 65.03% |
| trustworthy? | no | yes |

Three things to carry forward.

**A number needs a baseline before it means anything.** 93% sounds worse than 100% and is
enormously better. You cannot tell without knowing what guessing would have got you.

**Look inside the model.** The coefficients are how you tell a plausible story from a true
one. In Lab 1 a plausible explanation — that the model memorised the attacker's IP — was
impossible, because that column had been dropped. Only the coefficients showed what really
happened.

**The size of a data defect cannot be reasoned out in advance.** You removed 13 dead columns
and 1,438 duplicate rows and the score did not move. That does not mean audits are pointless;
it means the *magnitude* of a defect is an empirical question. Module 3 shows the same family
of problem turning an honest F1 of about 0.82 into a reported 0.95, and Module 5 shows one costing an entire class.

---

## Reflection Questions

**1.** *(Stretch.)* Of the attack types reviewed at the start of this module, which would be hardest to
detect with rule-based tools alone, and what specifically would machine learning add?

**[insert response]**

**2.** You are asked to build detection for a hospital network — medical devices, staff
workstations, patient records. Which of CICIDS2017, EMBER, IoT-23 or PhishTank would you
train on, and what would still be missing?

**[insert response]**

**3.** The Lab 1 model used `firewall_rule` — the firewall's own decision — as a feature. Name
another feature you might find in a real security dataset that leaks the answer in the same
way, and say how you would spot it.

**[insert response]**

**4.** Lab 2's model reached 93.35%. Roughly 7% of URLs are classified wrongly. For a phishing
blocker, which mistake is worse: blocking a legitimate site, or letting a phishing site
through? Justify it, then say how you would change the model's behaviour to reflect your
answer.

**[insert response]**

**5.** Module 1 audited the URL dataset and Module 2 acted on it — and the cleanup changed the
score by almost nothing. Write down what you would tell a colleague who concludes from this
that data auditing is overrated.

**[insert response]**

---

## Reference

The commands that mattered most in this module: `.drop_duplicates()`, `.dropna()`,
`.fillna()`, `pd.to_datetime()`, `.astype()`, `.str.capitalize()`, `pd.get_dummies()`,
`train_test_split()`, `MinMaxScaler` / `StandardScaler`, `LogisticRegression`,
`classification_report()` and `confusion_matrix()`.

* [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
* [scikit-learn: preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)
* [scikit-learn algorithm chooser](https://scikit-learn.org/stable/machine_learning_map.html)

## Sources and further reading

**Frameworks**
- MITRE ATT&CK, <https://attack.mitre.org/>.
- Hutchins, Cloppert and Amin, *Intelligence-Driven Computer Network Defense Informed by
  Analysis of Adversary Campaigns and Intrusion Kill Chains*, Lockheed Martin, 2011. The
  paper that introduced the Cyber Kill Chain.
  <https://www.lockheedmartin.com/en-us/capabilities/cyber/cyber-kill-chain.html>

**Datasets named in Section 3**
- Sharafaldin, Lashkari and Ghorbani, *Toward Generating a New Intrusion Detection Dataset
  and Intrusion Traffic Characterization* (CICIDS2017), ICISSP 2018.
- Anderson and Roth, *EMBER: An Open Dataset for Training Static PE Malware Machine Learning
  Models*, arXiv:1804.04637 (2018).
- Garcia, Parmisano and Erquiaga, *IoT-23: A labeled dataset with malicious and benign IoT
  network traffic*, Stratosphere Laboratory, 2020.
- PhishTank, <https://phishtank.org/>.

**Dataset used in Lab 2**
- Vrbančič, Fister and Podgorelec, *Datasets for phishing websites detection*, Data in Brief
  33 (2020).

**Tools**
- scikit-learn user guide: logistic regression
  <https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression> and model
  evaluation <https://scikit-learn.org/stable/modules/model_evaluation.html>.